# Oxford Pets Image Classification

**Deep Learning and Big Data Final Project**

In this notebook, we work with the Oxford-IIIT Pet dataset and build image classification models using PyTorch.

The project has two main goals:

1. Classify pet breeds using 37 classes.
2. Classify images as cat or dog.

We compare two approaches:

- a CNN trained from scratch
- a ResNet18 transfer learning model

The notebook shows the dataset, model setup, training process, evaluation results, and the final figures used in our report and presentation.


## 0. Setup

Run this cell first to import the libraries used in the notebook.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

# Import project files from src/oxpets.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "oxpets").exists() and (PROJECT_ROOT / "oxford_pets_project").exists():
    PROJECT_ROOT = PROJECT_ROOT / "oxford_pets_project"

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from oxpets.config import FIGURES_DIR, METRICS_DIR, TrainConfig, ensure_output_dirs, select_device, set_seed
from oxpets.data import TASKS, make_loaders
from oxpets.metrics import compute_metrics, save_confusion_matrix, save_history_plot
from oxpets.models import build_scratch_model, build_transfer_model
from oxpets.train import evaluate, train_model, trainable_parameters

ensure_output_dirs()
set_seed(42)

device = select_device()
print("Using device:", device)
print("Project folder:", PROJECT_ROOT.name)


## 1. Load Oxford Pets

We start with the cat vs dog task to show the complete training process clearly.


In [ ]:
config = TrainConfig(image_size=128, batch_size=16, num_workers=0)

train_loader, val_loader, test_loader, spec = make_loaders(
    task="species",
    config=config,
    download=True,
    limit_train=512,
    limit_val=128,
    limit_test=256,
)

print("Classes:", spec.class_names)
print("Training images:", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print("Test images:", len(test_loader.dataset))


## 2. Show Images

Before training, we look at a few images from the dataset.


In [ ]:
images, labels = next(iter(train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

plt.figure(figsize=(10, 5))
for i in range(8):
    image = (images[i] * std + mean).clamp(0, 1)
    image = image.permute(1, 2, 0)

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(spec.class_names[int(labels[i])])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 3. Check the Models

The output shape should be `(batch size, number of classes)`. For cat vs dog, the number of classes is 2.


In [ ]:
scratch_model = build_scratch_model(num_classes=spec.num_classes)
transfer_model = build_transfer_model(num_classes=spec.num_classes, pretrained=False)

with torch.no_grad():
    scratch_output = scratch_model(images)
    transfer_output = transfer_model(images)

print("Scratch CNN output:", scratch_output.shape)
print("Transfer model output:", transfer_output.shape)


## 4. Train a Small CNN

We train the scratch CNN for five epochs and then evaluate it on a separate test set.


In [ ]:
model = build_scratch_model(num_classes=spec.num_classes).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(trainable_parameters(model), lr=0.001, weight_decay=0.0001)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=loss_function,
    optimizer=optimizer,
    device=device,
    epochs=5,
)


## 5. Test the Model

Now we test the trained model on images it did not see during training.


In [ ]:
test_loss, test_accuracy, y_true, y_pred = evaluate(model, test_loader, loss_function, device)
metrics = compute_metrics(y_true, y_pred, spec.class_names)

print("Test accuracy:", round(test_accuracy, 3))
print("Macro F1:", round(metrics["macro_f1"], 3))


## 6. Plot Training Results

A loss curve and confusion matrix help us understand the result better than only one accuracy number.


In [ ]:
run_name = "notebook_species_scratch"

history_figure = FIGURES_DIR / f"{run_name}_history.png"
confusion_figure = FIGURES_DIR / f"{run_name}_confusion_matrix.png"

pd.DataFrame(history).to_csv(METRICS_DIR / f"{run_name}_history.csv", index=False)
save_history_plot(history, history_figure, "Species CNN training")
save_confusion_matrix(y_true, y_pred, spec.class_names, confusion_figure, "Species confusion matrix")

for figure in [history_figure, confusion_figure]:
    img = plt.imread(figure)
    plt.figure(figsize=(7, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


## 7. Final Project Figures

These figures come from the project experiments and are used in the report and presentation.


In [ ]:
figures = [
    "dataset_sample_grid.png",
    "augmentation_preview.png",
    "species_transfer_prediction_grid.png",
    "model_comparison_summary.png",
]

for name in figures:
    path = FIGURES_DIR / name
    if path.exists():
        img = plt.imread(path)
        plt.figure(figsize=(9, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(name.replace("_", " ").replace(".png", ""))
        plt.show()
    else:
        print("Figure not found:", name)


## 8. Project Files

The report and presentation are included in the project folder.

- `reports/Project_Report.pdf`
- `slides/Project_Presentation.pdf`
